# 02 — Normalize and QC: Canonical Phase-2 Population Execution

이 노트북은 `KOEN-TP-RS-001`과 P2 minimal-QC v1.1 계약을 변경 없이 실행한다. 출력은 진행상태, 안전한 집계, hash, 경로, 검증상태로 제한하며 KO/EN 원문·pair ID·복원 가능한 표본을 표시하지 않는다.

## 0 Contract / Scope

Population 실행은 본 배정에서 명시적으로 승인됐다. R1 수동감사 배정과 CR-003 문서 정리는 실행 blocker가 아니다. 모델 LID, NER, morphology, tokenizer 측정, embedding/semantic model은 이 단계에서 금지한다.

In [ ]:
FULL_RUN_AUTHORIZED = True
RESEARCH_SEMANTICS_FROZEN = True
MANUAL_AUDIT_SAMPLE_DRAW_AUTHORIZED = False
FORBIDDEN_COMPONENTS = {
    'lingua', 'fasttext', 'langid', 'transformer_lid', 'embedding_similarity',
    'ner', 'morphology', 'o200k_token_count', 'tp', 'logtp', 'modeling',
}
assert FULL_RUN_AUTHORIZED and RESEARCH_SEMANTICS_FROZEN
assert not MANUAL_AUDIT_SAMPLE_DRAW_AUTHORIZED

## 1 Input lineage

전용 worktree의 immutable D-01 복제본과 추적 중인 manifest를 사용한다. 변환 전에 실제 파일 SHA-256을 manifest 값과 대조한다.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd
import psutil
import pyarrow.parquet as pq

from tokenization_premium.paths import PROJECT_ROOT
from tokenization_premium.phase2 import (
    EXPECTED_D01_ROW_COUNT,
    P2_CONTRACT_COMMIT,
    P2_DUCKDB_MEMORY_LIMIT,
    P2_THREADS,
    PAIR_REGISTRY_V001_RELATIVE_PATH,
    PAIR_REGISTRY_V002_RELATIVE_PATH,
    execute_phase2_population,
    validate_d01_manifest_handoff,
)

os.environ['TOKENIZATION_PREMIUM_DUCKDB_MEMORY_LIMIT'] = '6GB'
os.environ['TOKENIZATION_PREMIUM_PROGRESS_INTERVAL_SEC'] = '10'
os.environ['TOKENIZATION_PREMIUM_PROGRESS_DIR'] = '.runtime/progress'

PAIR_REGISTRY_V001 = PROJECT_ROOT / PAIR_REGISTRY_V001_RELATIVE_PATH
PAIR_REGISTRY_V002 = PROJECT_ROOT / PAIR_REGISTRY_V002_RELATIVE_PATH
D01_MANIFEST_PATH = PROJECT_ROOT / 'outputs/manifests/PAIR_REGISTRY_MANIFEST_v001.json'
D01_ORACLE_PATH = PROJECT_ROOT / 'outputs/manifests/data_recon/G1_INGEST_EXPECTATIONS_v001.json'
CONTRACT_PATH = PROJECT_ROOT / 'docs/contracts/P2_NORMALIZE_QC_PRECONTRACT_v1.md'
RUNTIME_DIR = PROJECT_ROOT / '.runtime/p2-canonical-run'

available_gib = psutil.virtual_memory().available / 1024**3
swap_gib = psutil.swap_memory().total / 1024**3
disk_gib = psutil.disk_usage(PROJECT_ROOT).free / 1024**3
heavy_processes = []
for process in psutil.process_iter(['pid', 'name', 'cmdline', 'memory_info']):
    try:
        if process.info['pid'] == os.getpid():
            continue
        command = ' '.join(process.info.get('cmdline') or []).lower()
        rss_gib = process.info['memory_info'].rss / 1024**3 if process.info.get('memory_info') else 0
        if 'duckdb' in command or ('tokenization_premium' in command and 'python' in command and rss_gib >= 2):
            heavy_processes.append({'pid': process.info['pid'], 'rss_gib': round(rss_gib, 3)})
    except (psutil.AccessDenied, psutil.NoSuchProcess):
        pass

assert available_gib >= 8, f'available RAM gate failed: {available_gib:.2f} GiB'
assert swap_gib > 0, 'swap gate failed'
assert disk_gib >= 50, f'disk gate failed: {disk_gib:.2f} GiB'
assert not heavy_processes, f'other heavy project process detected: {heavy_processes}'
assert PAIR_REGISTRY_V001.is_file()
assert P2_DUCKDB_MEMORY_LIMIT == '6GB' and P2_THREADS == 4
{
    'input_path': str(PAIR_REGISTRY_V001.relative_to(PROJECT_ROOT)),
    'expected_rows': EXPECTED_D01_ROW_COUNT,
    'available_ram_gib': round(available_gib, 3),
    'swap_gib': round(swap_gib, 3),
    'disk_free_gib': round(disk_gib, 3),
    'other_heavy_processes': len(heavy_processes),
    'duckdb_memory_limit': P2_DUCKDB_MEMORY_LIMIT,
    'threads': P2_THREADS,
    'spill_directory': str((RUNTIME_DIR / 'duckdb-spill').relative_to(PROJECT_ROOT)),
}

## 2 D-01 handoff verification

D-01 manifest PASS, 5,652,925행, pair-ID cardinality, canonical input allowlist와 raw SHA linkage를 먼저 검증한다. 이 단계에서는 population text를 읽지 않는다.

In [ ]:
d01_manifest = json.loads(D01_MANIFEST_PATH.read_text(encoding='utf-8'))
d01_oracle = json.loads(D01_ORACLE_PATH.read_text(encoding='utf-8'))
d01_handoff = validate_d01_manifest_handoff(d01_manifest, d01_oracle)
assert d01_handoff.status == 'PASS'
assert d01_handoff.row_count == EXPECTED_D01_ROW_COUNT
assert P2_CONTRACT_COMMIT == 'b9990afbf3fc0ed2a5e80fb4def1565e9ba3ebf4'
{
    'handoff_status': d01_handoff.status,
    'manifest_rows': d01_handoff.row_count,
    'pair_id_distinct': d01_handoff.pair_id_distinct,
    'canonical_input_count': d01_handoff.canonical_input_count,
    'manifest_input_sha256': d01_manifest['pair_registry']['sha256'],
    'contract_commit': P2_CONTRACT_COMMIT,
}

## 3 Normalization

`execute_phase2_population`이 실제 input SHA 일치를 먼저 확인한 뒤, 전체 행에 NFC → edge-only U+FEFF 제거 → outer whitespace trim을 적용한다. raw와 내부 whitespace/U+FEFF는 보존된다. 동일 atomic 실행에서 이후 QC 컬럼도 계산한다.

In [ ]:
run_id = 'P2_CANONICAL_' + datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%dT%H%M%S')
population_result = execute_phase2_population(
    project_root=PROJECT_ROOT,
    input_path=PAIR_REGISTRY_V001,
    output_path=PAIR_REGISTRY_V002,
    d01_manifest_path=D01_MANIFEST_PATH,
    contract_path=CONTRACT_PATH,
    runtime_dir=RUNTIME_DIR,
    run_id=run_id,
)
{
    'run_id': population_result.run_id,
    'normalization_status': 'PHASE2_COMPLETE',
    'row_count': population_result.row_count,
    'input_sha256': population_result.input_sha256,
    'validation_status': population_result.validation_status,
}

## 4 Decode / Unicode integrity

Unicode replacement character와 decode-integrity 문제는 structural gate로, raw 기반 zero-width/control/orphan-combining 관측은 별도 advisory aggregate로 보고한다. tokenizer roundtrip은 수행하지 않는다.

In [ ]:
{
    'decode_integrity_flag': population_result.qc_counts['decode_integrity_flag'],
    'unicode_anomaly_flag_review_only': population_result.qc_counts['unicode_anomaly_flag'],
    'tokenizer_roundtrip': 'NOT_RUN_G3_SCOPE',
}

## 5 Structural QC

동결된 5개 hard flag만 population rejection을 결정한다. Advisory flag는 rejection 원인이 아니다.

In [ ]:
structural_qc_counts = {
    name: population_result.qc_counts[name]
    for name in (
        'empty_text_flag', 'decode_integrity_flag', 'markup_dominant_flag',
        'control_char_excess_flag', 'exact_duplicate_flag',
    )
}
advisory_qc_counts = {
    name: population_result.qc_counts[name]
    for name in (
        'unicode_anomaly_flag', 'high_digit_ratio_flag',
        'high_punctuation_ratio_flag', 'script_mix_flag',
    )
}
{'structural': structural_qc_counts, 'advisory_review_only': advisory_qc_counts}

## 6 Exact duplicate analysis disposition

D-01 `representative_pair_id`는 provenance pointer로 그대로 둔다. 별도 `analysis_representative_pair_id`는 025/026 eligible row를 우선한 분석 생존자이며, row는 삭제하지 않는다.

In [ ]:
{
    'raw_rows_preserved': population_result.qc_counts['raw_record_denominator'],
    'exact_unique_content_denominator': population_result.qc_counts['exact_unique_content_denominator'],
    'non_representative_exact_duplicates': population_result.qc_counts['exact_duplicate_flag'],
    'provenance_pointer': 'UNCHANGED',
    'analysis_survivor_field': 'analysis_representative_pair_id',
    'near_duplicate_claim': False,
}

## 7 Language-side sanity review

역사적 report 이름의 LID는 language-side sanity / SSOT traceability를 뜻한다. model-based language identification이 아니며, reason은 side별로 분리하고 자동 reject하지 않는다.

In [ ]:
{
    'method': 'deterministic Unicode script-side sanity; review only',
    'model_based_lid': False,
    'automatic_rejection': False,
    'review_flag_total': population_result.qc_counts['lang_side_anomaly_review_flag'],
    'reason_counts_by_side': dict(population_result.language_side_counts),
}

## 8 Population pair_quality_status

5개 structural flag 중 하나라도 true면 rejected, 모두 false면 accepted다. 미감사 row를 review로 강등하지 않는다.

In [ ]:
{
    'accepted': population_result.qc_counts['accepted'],
    'rejected': population_result.qc_counts['rejected'],
    'total': population_result.qc_counts['raw_record_denominator'],
    'manual_audit_gate': False,
}

## 9 QC aggregate flow

4개 denominator와 모든 flag/disposition을 raw-text-free 집계로 저장한다.

In [ ]:
qc_flow_path = PROJECT_ROOT / 'outputs/reports/QC_FLOW_v001.csv'
lid_report_path = PROJECT_ROOT / 'outputs/reports/LID_QC_PASS_RATE_v001.csv'
sampling_frame_path = PROJECT_ROOT / 'outputs/reports/MANUAL_QC_SAMPLING_FRAME_SUMMARY_v001.csv'
qc_flow = pd.read_csv(qc_flow_path)
assert set(qc_flow['metric']) >= {
    'raw_record_denominator', 'exact_unique_content_denominator',
    'primary_eligible_denominator', 'final_analysis_denominator',
}
qc_flow

## 10 v002 artifact validation

동일 5,652,925행, unique pair_id, raw immutability, 필수 P2 컬럼, atomic promotion, partial 부재를 검증한 최종 artifact만 노출한다.

In [ ]:
v002_parquet = pq.ParquetFile(PAIR_REGISTRY_V002)
assert v002_parquet.metadata.num_rows == EXPECTED_D01_ROW_COUNT
assert not PAIR_REGISTRY_V002.with_suffix(PAIR_REGISTRY_V002.suffix + '.partial').exists()
{
    'path': str(PAIR_REGISTRY_V002.relative_to(PROJECT_ROOT)),
    'rows': v002_parquet.metadata.num_rows,
    'columns': len(v002_parquet.schema_arrow.names),
    'sha256': population_result.output_sha256,
    'raw_columns_unchanged': True,
    'row_deletion': False,
    'partial_remaining': False,
    'validation_status': population_result.validation_status,
}

## 11 Manifest / hashes / G1 evidence summary

manifest, 안전 집계 report hash, execution timing/resource evidence를 요약한다. 500쌍 raw human sample은 추출하지 않았고 R1 logistics는 pending이다.

In [ ]:
manifest_path = PROJECT_ROOT / 'outputs/manifests/QC_MANIFEST_v001.json'
execution_report_path = PROJECT_ROOT / 'outputs/reports/P2_EXECUTION_REPORT_v001.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
safe_paths = [qc_flow_path, lid_report_path, sampling_frame_path, execution_report_path, manifest_path]
safe_hashes = {
    str(path.relative_to(PROJECT_ROOT)): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in safe_paths
}
{
    'manifest_validation': manifest['validation_status'],
    'execution_code_commit': manifest['execution_code_commit'],
    'start_kst': manifest['start_kst'],
    'end_kst': manifest['end_kst'],
    'stage_durations_sec': manifest['stage_durations_sec'],
    'resource_observation': manifest['resource_observation'],
    'safe_report_hashes': safe_hashes,
    'manual_audit_status': manifest['manual_audit_status'],
    'g1_final_adjudication': 'PENDING',
}